# 3) EE-2.1 AC, phasors and impedance — Exercises

**Goals:** RMS values, phasor arithmetic, impedances at different frequencies, series circuits and resonance, checking phasors against a time-domain simulation, and how a line's X/R ratio decides whether power flow shows up in voltage magnitudes or in voltage angles.

All AC magnitudes are RMS values. Use `1j` for the imaginary unit. Do task 4's resonance condition on paper first.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import cmath
import math

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

F = 50.0                       # Hz

### Warm-ups

**1. RMS from samples**

Write `rms(samples)`. Check that a sinusoid with a 325 V peak gives ≈ 230 V — and that a square wave of ±100 V gives 100 V, **not** 100/√2: the √2 rule holds for sinusoids only.

In [ ]:
def rms(samples) -> float:
    """Root mean square of equally spaced samples covering whole periods."""
    ...  # YOUR CODE HERE

In [ ]:
t = np.linspace(0, 1 / F, 100_000, endpoint=False)
assert math.isclose(rms(325 * np.cos(2 * np.pi * F * t)), 325 / math.sqrt(2), rel_tol=1e-9)
square = np.where(np.sin(2 * np.pi * F * t) >= 0, 100.0, -100.0)
assert math.isclose(rms(square), 100.0), "a square wave's RMS equals its amplitude"

**2. Phasors in rectangular and polar form**

Write `phasor(rms_value, angle_deg)` returning a complex number, and `polar(z)` returning `(magnitude, angle_deg)`.

In [ ]:
def phasor(rms_value: float, angle_deg: float) -> complex:
    """Complex phasor from RMS magnitude and angle in degrees."""
    ...  # YOUR CODE HERE


def polar(z: complex) -> tuple[float, float]:
    """(magnitude, angle in degrees) of a phasor."""
    ...  # YOUR CODE HERE

In [ ]:
assert np.isclose(phasor(230, 0), 230) and np.isclose(phasor(10, 90), 10j)
mag, ang = polar(phasor(120, -30))
assert math.isclose(mag, 120) and math.isclose(ang, -30)
mag, ang = polar(phasor(230, 0) + phasor(230, -120))     # two phases of a three-phase system
assert math.isclose(mag, 230) and math.isclose(ang, -60), "120 degrees apart: the sum has the same magnitude"

### Core

**3. Impedances at different frequencies**

Write `z_inductor(L, f)` and `z_capacitor(C, f)`. Then answer with the tests: by what factor does each reactance change from 50 Hz to 60 Hz?

In [ ]:
def z_inductor(L: float, f: float = F) -> complex:
    """Impedance (ohm) of an inductance L (H) at frequency f (Hz)."""
    ...  # YOUR CODE HERE


def z_capacitor(C: float, f: float = F) -> complex:
    """Impedance (ohm) of a capacitance C (F) at frequency f (Hz)."""
    ...  # YOUR CODE HERE

In [ ]:
assert np.isclose(z_inductor(0.05), 15.70796j, atol=1e-5)
assert np.isclose(z_capacitor(200e-6), -15.91549j, atol=1e-5)
assert np.isclose(z_inductor(0.05, 60) / z_inductor(0.05, 50), 1.2), "inductive reactance grows with frequency"
assert np.isclose(z_capacitor(200e-6, 60) / z_capacitor(200e-6, 50), 1 / 1.2), "capacitive reactance falls"

**4. Series circuits and resonance**

Write `series_circuit(V, impedances)`: the current phasor and the voltage phasor across every element. Check KVL. Then find the capacitance that makes a series R–L–C circuit **resonate** at 50 Hz (inductive and capacitive reactance cancel), and look at the voltage across the inductor.

In [ ]:
def series_circuit(V: complex, impedances: list[complex]) -> tuple[complex, list[complex]]:
    """Current phasor (A) and element voltage phasors (V) of a series circuit."""
    ...  # YOUR CODE HERE


def resonant_capacitance(L: float, f: float = F) -> float:
    """Capacitance (F) that resonates with L at frequency f."""
    ...  # YOUR CODE HERE

In [ ]:
I, (vR, vL) = series_circuit(230, [10, z_inductor(0.05)])
assert np.isclose(vR + vL, 230), "KVL holds for phasors"
assert abs(vR) + abs(vL) > 230, "magnitudes do not add up"

C_res = resonant_capacitance(0.05)
I_res, (vR, vL, vC) = series_circuit(230, [10, z_inductor(0.05), z_capacitor(C_res)])
assert np.isclose(I_res, 23.0), "at resonance only R limits the current: 230 V / 10 ohm"
assert abs(vL) > 230, "at resonance the inductor voltage exceeds the supply"
print(f"C_res = {C_res * 1e6:.1f} uF, I = {abs(I_res):.1f} A, |V_L| = {abs(vL):.0f} V, |V_C| = {abs(vC):.0f} V")

**5. Check a phasor against the differential equation**

Write `steady_state_current(R, L, V_rms, f)`: simulate L·di/dt + R·i = √2·V_rms·cos(ωt) from rest with `solve_ivp`, discard the transient (simulate at least 10 time constants L/R and 10 periods), fit amplitude and phase over the last period, and return the current as an **RMS phasor**. It must match V/Z.

In [ ]:
def steady_state_current(R: float, L: float, V_rms: float, f: float = F) -> complex:
    """Steady-state current phasor (RMS) of a series RL circuit, from a time-domain simulation."""
    ...  # YOUR CODE HERE

In [ ]:
for R_, L_ in [(5.0, 0.1), (20.0, 0.01)]:
    simulated = steady_state_current(R_, L_, 230)
    analytic = 230 / (R_ + z_inductor(L_))
    assert np.isclose(simulated, analytic, rtol=1e-5), f"R={R_}, L={L_}: simulation and phasor disagree"
    print(f"R={R_:4.1f} ohm, L={L_*1e3:5.1f} mH: simulated {abs(simulated):.5f} A at {np.angle(simulated, deg=True):7.3f} deg"
          f" | phasor {abs(analytic):.5f} A at {np.angle(analytic, deg=True):7.3f} deg")

**6. Interpretation**

At resonance in task 4, the voltage across the inductor is several times the supply voltage, although the supply delivers only 230 V. Where does that voltage come from, why is it a hazard in real networks (think of long cables, which are capacitive, connected to inductive equipment), and why did the magnitudes in the RL circuit not add up to 230 V either?

*Your answer here.*

### Challenge

**7. Does the line lower the voltage or turn it?**

A 230 V source feeds a purely resistive 20 Ω load through a line. Compare three lines with the **same impedance magnitude** |Z| = 2 Ω: purely resistive, a typical overhead line with X/R = 4, and purely reactive. Write `line_effect(z_line)` returning the load voltage magnitude (V), its angle (degrees) and the magnitude drop (V). Use the complex `solve_fixed_ac` from the lecture or a voltage divider.

In [ ]:
def line_effect(z_line: complex, z_load: complex = 20.0, V_source: complex = 230.0) -> dict:
    """Load voltage magnitude, angle and magnitude drop for a source -> line -> load circuit."""
    ...  # YOUR CODE HERE

In [ ]:
lines = {"pure R": 2.0 + 0j,
         "overhead line X/R = 4": 2.0 * cmath.exp(1j * math.atan(4)),
         "pure X": 2.0j}
table = pd.DataFrame({name: line_effect(z) for name, z in lines.items()}).T
assert np.allclose([abs(z) for z in lines.values()], 2.0), "same impedance magnitude for all three"
assert abs(table.at["pure R", "angle_deg"]) < 1e-9, "a resistive line does not turn the voltage"
assert table["magnitude_drop_V"].is_monotonic_decreasing, "the more reactive, the smaller the magnitude drop"
assert table["angle_deg"].abs().is_monotonic_increasing, "the more reactive, the larger the angle shift"
table.round(2)

**8. Interpretation**

Transmission lines have X/R ratios of roughly 5–10 or more. Based on your table, what happens mostly to the voltage at the receiving end when such a line carries active power to a resistive load — its magnitude or its angle? Why does that make voltage **angles** the natural variable for describing active power flows in the DC power flow approximation of Module 04?

*Your answer here.*